# NB_patient_360
**Target:** `caresyncdatabricks.gold.patient_360`  |  **Grain:** one row per patient  |  **Load:** FULL (CREATE OR REPLACE TABLE ... AS SELECT)

**Driving table:** `silver.patients` (no filters, all patients kept)
* LEFT JOIN `silver.hospitals` ON `registered_hospital_id = hospital_id`
* LEFT JOIN `silver.insurance_providers` ON `insurance_provider_id`
* LEFT JOIN `silver.lab_results` aggregated by `patient_id` (counts default to 0, `last_test_date` stays NULL when the patient has no tests)

Note: `silver.patients.date_of_birth` is stored as a timestamp, so it is cast to `date` here to match the STTM.

In [0]:
%sql
CREATE OR REPLACE TABLE caresyncdatabricks.gold.patient_360 AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                             AS total_tests,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count,
    MAX(test_date)                       AS last_test_date
  FROM caresyncdatabricks.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                             AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                         AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT)  AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                                 AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                                  AS insurance_provider_id,
  ip.provider_name                                                                      AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                                              AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                                      AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                                      AS critical_test_count,
  la.last_test_date
FROM caresyncdatabricks.silver.patients p
LEFT JOIN caresyncdatabricks.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN caresyncdatabricks.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id

In [0]:
updated_df = spark.table("caresyncdatabricks.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))